# Optimized LLM Fine Tuning Experiment

My sufficient statistic regularizer (SSR) framework and associated theory provides optimal statistical efficiency guarantees.
The applied impact of improved statistical efficiency must be observed; it can't be derived. 
In this experiment, I try to see if something new is possible: LLM tuning with very small datasets. 
For example, could an agent learn from a single user? 
Can information be retained in model parameters so retrieval isn't limited by context windows? 
We have to try and see. 

In [3]:
!ip a

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 00:0d:3a:1f:94:a2 brd ff:ff:ff:ff:ff:ff
    inet 10.0.0.5/19 brd 10.0.31.255 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::20d:3aff:fe1f:94a2/64 scope link 
       valid_lft forever preferred_lft forever
3: docker0: <NO-CARRIER,BROADCAST,MULTICAST,UP> mtu 1500 qdisc noqueue state DOWN group default 
    link/ether 02:42:0c:e4:d6:57 brd ff:ff:ff:ff:ff:ff
    inet 172.17.0.1/16 brd 172.17.255.255 scope global docker0
       valid_lft forever preferred_lft forever


[W228 18:57:51.055927929 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:07.164369539 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3


In [5]:
## hacking torch.distributed to run on notebooks 

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import os

def all_reduce_example(rank, world_size):
    """Simple all_reduce example across multiple CPU processes."""
    # notebook hacking: inject torch.distributed envs 
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29500'
    os.environ["GLOO_SOCKET_IFNAME"] = "eth0"

    # Initialize the process group
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size)
    
    # Each process starts with a tensor of its rank
    tensor = torch.tensor([rank], dtype=torch.float32)
    
    print(f"Before all_reduce - Rank {rank}: {tensor.item()}")
    
    # Perform all_reduce (sum operation)
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    
    print(f"After all_reduce - Rank {rank}: {tensor.item()}")
    
    # Cleanup
    dist.destroy_process_group()

def notebook_run(func, args, nprocs):
    processes = []
    for rank in range(nprocs):
        p = mp.Process(target=func, args=(1, *args))
        p.start()
        processes.append(p)
        pass 
    for p in processes:
        p.join()
        pass 
    pass 


world_size = 4  # Number of processes
## doesn't work in notebook envs 
##torch.multiprocessing.spawn(all_reduce_example, args=(world_size,), nprocs=world_size, join=True)
notebook_run(all_reduce_example, args=(world_size,), nprocs=world_size) 

[W228 18:58:34.672727936 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:34.685189466 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:34.698637698 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:34.709587524 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:35.933674161 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:35.094554946 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:35.108428180 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:35.246076309 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:58:36.951683000 socket.cpp:204] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W228 18:5

KeyboardInterrupt: 

In [1]:
## learning a little torch distributed computing... 

import torch
import torch.distributed as dist
import torch.multiprocessing as mp 

def all_reduce_example(rank, max_val, world_size, mp_queue):
    """Simple all_reduce example across multiple CPU processes."""
    ## initialize the process group
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size) 
    ## perform sum 
    sum_for_this_rank = torch.concat([list(range(max_val))[i] for i in len(max_val) if i % rank == 0]).sum().float() 
    dist.all_reduce(sum_for_this_rank, op=dist.ReduceOp.SUM) 
    ## return 
    if rank == 0: 
        mp_queue.put(sum_for_this_rank.item().float()) 
    ## cleanup 
    dist.destroy_process_group() 
    pass  

if __name__ == '__main__': 
    ## use 4 processes 
    world_size = n_procs = 4 
    ## output queue 
    mp_queue = mp.Queue() 
    ## parallel compute 
    torch.multiprocessing.spawn(all_reduce_example, args=(10, world_size, mp_queue), nprocs=world_size, join=True) 
    ## print output 
    print(f'distributed calculation: {mp_queue.get()}')
    tensor_list = [torch.tensor(i) for i in range(10)]  
    print(f'local calculation: {torch.cat(tensor_list).sum()}')  

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/anaconda/envs/azureml_py38/lib/python3.8/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/anaconda/envs/azureml_py38/lib/python3.8/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'all_reduce_example' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/anaconda/envs/azureml_py38/lib/python3.8/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/anaconda/envs/azureml_py38/lib/python3.8/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'all_reduce_example' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/anaconda/envs/azu

ProcessExitedException: process 0 terminated with exit code 1

In [1]:
import torch
import torch.distributed as dist
import os

def all_reduce_example(rank, world_size):
    """Simple all_reduce example across multiple CPU processes."""
    # Initialize the process group
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size)
    
    # Each process starts with a tensor of its rank
    tensor = torch.tensor([rank], dtype=torch.float32)
    
    print(f"Before all_reduce - Rank {rank}: {tensor.item()}")
    
    # Perform all_reduce (sum operation)
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    
    print(f"After all_reduce - Rank {rank}: {tensor.item()}")
    
    # Cleanup
    dist.destroy_process_group()

if __name__ == "__main__":
    world_size = 4  # Number of processes
    torch.multiprocessing.spawn(all_reduce_example, args=(world_size,), nprocs=world_size, join=True)


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'all_reduce_example' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'all_reduce_example' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <mo

ProcessExitedException: process 2 terminated with exit code 1